# 10 - MF4 / Video Alignment EDA  (Track B - optional enhancement)

**This notebook is NOT part of the detector pipeline (01-08).** The trained
model needs no distance signal. Track B explores adding *metric* information -
geometric object height and camera/USS fusion - on top of the existing
detector, and this EDA answers the prerequisite question:

> Does the MF4 <-> video alignment recipe hold across the whole dataset,
> or only on the one sequence where it was reverse-engineered?

The recipe (see `docs/mf4_video_alignment.md`): the MF4 carries a camera
channel per view (`WebCam` front / `WebCam3` rear) whose **sample values are the
video frame PTS** and whose **timestamps are MF4 master time** - an exact clock
pairing. USS distance channels live on the same clock.

**Sampling, not a full scan.** MF4 files are ~90 MB each; reading all ~8k
sequences would pull ~700 GB from Drive. This samples `N_SAMPLE` random
sequences (seeded), which is plenty to establish whether the convention is
universal. Resumable and parallel; results append to a CSV on Drive.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "asammdf"], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import csv
import json as _json
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm.auto import tqdm

from vision_uss_research.alignment.mf4 import camera_for_direction, probe_sequence
from vision_uss_research.runs import finish_run, new_run_id, resolve_run, start_run
from vision_uss_research.sequences import (driving_direction, sequence_files,
                                           with_drive_retry)

N_SAMPLE = 150        # sequences to probe (0 = all; ~90 MB each, so beware)
SEED = 0
NUM_WORKERS = 8       # MF4 metadata parse is Drive-read bound
INVENTORY_RUN = None  # None = newest non-smoke inventory run

RUN_DIR = ARTIFACTS / "alignment" / new_run_id("alignment", "eda")
RESULTS_CSV = RUN_DIR / "alignment_probe.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)

inv_dir = resolve_run(ARTIFACTS / "inventory", "inventory", run_id=INVENTORY_RUN,
                      require_file="inventory_folders.csv", exclude_suffix="_smoke")
assert inv_dir is not None, "no inventory run - run notebook 01 first"
inv = pd.read_csv(inv_dir / "inventory_folders.csv")
usable = inv[(inv["status"] == "complete")
             & ((inv["has_front_video"] == 1) | (inv["has_rear_video"] == 1))]

rng = random.Random(SEED)
seq_ids = sorted(usable["sequence_id"].tolist())
if N_SAMPLE and N_SAMPLE < len(seq_ids):
    seq_ids = sorted(rng.sample(seq_ids, N_SAMPLE))

done = set()
if RESULTS_CSV.exists():
    done = set(pd.read_csv(RESULTS_CSV)["sequence_id"].astype(str))
    print(f"resuming: {len(done)} sequences already probed")
todo = [s for s in seq_ids if s not in done]

start_run(RUN_DIR, config={"n_sample": N_SAMPLE, "seed": SEED,
                           "inventory": inv_dir.name},
          inputs=[f"inventory/{inv_dir.name}"], repo_root=REPO_DIR)
print(f"inventory : {inv_dir.name}")
print(f"to probe  : {len(todo)} of {len(seq_ids)} sampled "
      f"(dataset has {len(usable)} complete sequences)")
print(f"output    : {RESULTS_CSV}")

## Probe (parallel, resumable - rerun this cell any time)

In [ ]:
FIELDS = ["sequence_id", "alignable", "error", "n_channels",
          "driving_direction", "required_camera", "required_camera_counts_match",
          "front_channel", "front_n_samples", "front_n_frames", "front_counts_match",
          "front_mf4_span_s", "front_pts_span_s", "front_pts_monotone",
          "rear_channel", "rear_n_samples", "rear_n_frames", "rear_counts_match",
          "rear_mf4_span_s", "rear_pts_span_s", "rear_pts_monotone",
          "pdc_channels_with_data", "n_objbuff_with_data"]

def probe_one(seq_id):
    seq_dir = RAW_ROOT / seq_id
    try:
        found = with_drive_retry(sequence_files, seq_dir)
        mf4 = next(iter(sorted(seq_dir.glob("*.mf4"))), None)
        if mf4 is None:
            return {"sequence_id": seq_id, "error": "no_mf4", "alignable": 0}
        videos = {}
        if found["front_video"] is not None:
            videos["front"] = found["front_video"]
        if found["rear_video"] is not None:
            videos["rear"] = found["rear_video"]
        # only the camera facing the approach matters (forward->front,
        # backward->rear); the other view never sees the object
        direction = with_drive_retry(driving_direction, seq_dir)
        required = camera_for_direction(direction)
        if required not in videos:
            required = None          # unknown direction -> judge on any camera
        row = probe_sequence(mf4, videos, required_camera=required)
        row["sequence_id"] = seq_id
        row["driving_direction"] = direction
        return row
    except Exception as e:
        return {"sequence_id": seq_id, "error": f"{type(e).__name__}:{e}",
                "alignable": 0}

def append_row(row):
    write_header = not RESULTS_CSV.exists()
    with RESULTS_CSV.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore")
        if write_header:
            w.writeheader()
        w.writerow(row)

if todo:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = [pool.submit(probe_one, s) for s in todo]
        for fut in tqdm(as_completed(futures), total=len(futures),
                        desc="Probing MF4s", unit="seq"):
            append_row(fut.result())

results = pd.read_csv(RESULTS_CSV)
print(f"probed: {len(results)} sequences")

## Verdict: does the alignment convention hold dataset-wide?

In [ ]:
n = len(results)
err = results["error"].fillna("")
no_mf4 = int((err == "no_mf4").sum())
other_err = int(((err != "") & (err != "no_mf4")).sum())
with_mf4 = results[err == ""]
n_align = int(results["alignable"].fillna(0).sum())

print(f"sequences probed              : {n}")
print(f"  no MF4 file present         : {no_mf4}  ({100*no_mf4/max(n,1):.1f}%)"
      f"   <- data availability, NOT an alignment failure")
print(f"  other open/probe errors     : {other_err}")
print(f"  MF4 present & readable      : {len(with_mf4)}")
print()
print(f"ALIGNMENT RATE among sequences that HAVE an MF4:")
print(f"  alignable                   : {n_align}/{len(with_mf4)}"
      f"  ({100*n_align/max(len(with_mf4),1):.1f}%)   <- the number that matters")
print(f"  (as a share of all probed)  : {100*n_align/max(n,1):.1f}%")

if "required_camera" in results.columns:
    rc = with_mf4["required_camera"].fillna("")
    print(f"\ndirection-relevant camera (only this one is needed per sequence):")
    print(f"  direction resolved          : {int((rc != '').sum())}/{len(with_mf4)}")
    for cam in ["front", "rear"]:
        sel = with_mf4[rc == cam]
        if len(sel):
            m = sel["required_camera_counts_match"].fillna(0)
            print(f"  {cam:<5} needed: {len(sel):>4}   counts match: "
                  f"{int(m.sum())}/{len(sel)}  ({100*m.mean():.1f}%)")

print("\nper-camera detail (informational - only the needed camera gates alignment):")
for cam in ["front", "rear"]:
    ch = results.get(f"{cam}_channel")
    if ch is None:
        continue
    has = ch.fillna("") != ""
    match = results.get(f"{cam}_counts_match").fillna(0)
    mono = results.get(f"{cam}_pts_monotone").fillna(0)
    print(f"  {cam}: channel present {int(has.sum())}/{len(with_mf4)} "
          f"(names: {sorted(set(ch[has].tolist()))}), "
          f"counts match {int(match.sum())}/{int(has.sum())}, "
          f"PTS monotone {int(mono.sum())}/{int(has.sum())}")

if results["error"].fillna("").ne("").any():
    print("\nerrors:")
    display(results[results["error"].fillna("") != ""]["error"]
            .str.slice(0, 60).value_counts().to_frame("n"))

In [ ]:
# Which USS distance channels are populated, and how often?
from collections import Counter
c = Counter()
for s in results["pdc_channels_with_data"].fillna(""):
    for part in str(s).split("|"):
        if part:
            c[part.split(":")[0]] += 1
if c:
    display(pd.Series(c).sort_values(ascending=False)
            .to_frame("sequences_with_readings"))
print("object-buffer slots with data (median per sequence):",
      results["n_objbuff_with_data"].median())

# Sequences where alignment would FAIL - the ones to investigate
# real alignment failures = MF4 present but the needed camera does not match
bad = results[(results["alignable"].fillna(0) == 0)
              & (results["error"].fillna("") == "")]
print(f"\nMF4 present but NOT alignable: {len(bad)}  (excludes the "
      f"{int((results['error'].fillna('') == 'no_mf4').sum())} with no MF4)")
if len(bad):
    display(bad[["sequence_id", "driving_direction", "required_camera",
                 "front_counts_match", "rear_counts_match",
                 "n_objbuff_with_data"]].head(20))
missing = results[results["error"].fillna("") == "no_mf4"]["sequence_id"]
if len(missing):
    print(f"\nsequences with no MF4 ({len(missing)}) - check whether these are "
          f"genuinely absent or just not synced to Drive:")
    for s in missing.head(10):
        print("   ", s)

In [ ]:
summary = {
    "probed": int(n),
    "no_mf4": no_mf4,
    "with_mf4": int(len(with_mf4)),
    "alignable": int(results["alignable"].fillna(0).sum()),
    "alignable_pct_of_with_mf4": round(
        100 * n_align / max(len(with_mf4), 1), 1),
    "alignable_pct_of_probed": round(100 * float(results["alignable"].fillna(0).mean()), 1),
    "front_counts_match": int(results.get("front_counts_match",
                                          pd.Series(dtype=float)).fillna(0).sum()),
    "rear_counts_match": int(results.get("rear_counts_match",
                                         pd.Series(dtype=float)).fillna(0).sum()),
    "errors": int(results["error"].fillna("").ne("").sum()),
}
finish_run(RUN_DIR, summary=summary)
print(_json.dumps(summary, indent=1))
print(f"\nresults: {RESULTS_CSV}")

## Reading the verdict

- **`sample count == frame count` near 100%** -> the pairing convention is
  universal; Track B can proceed to per-frame distance for the whole dataset.
- **High but not universal** -> proceed, but the height/fusion experiments run
  on the alignable subset; record that subset as the experiment population.
- **Low / erratic channel names** -> the convention is campaign-specific; fall
  back to the CusReplay `ObjDist` curves with a cycle->time bridge, or scope
  Track B to the sequences that do align.

Next in Track B (only after this passes): pick the distance channel/object that
tracks the labeled target, confirm units against the 2000 mm labeled event,
calibrate focal length from known-height classes (dummychild 125 cm, pole
108 cm), then emit `height = box_px_height * distance / focal` per detection.